# MERENVAY — ARC Prize 2026 ARC-AGI-2 offline submission\n\nCompetition wrapper for the answer-isolated candidate. Attach the MERENVAY offline bundle `.tar.gz` and its `.sha256` file as Kaggle input. The notebook verifies the bundle, extracts it inside `/kaggle/working`, runs with internet disabled, writes `submission.json`, and performs no grading. No ARC Prize Verified claim is implied until ARC Prize/Kaggle returns an official evaluation result.\n

In [ ]:
from pathlib import Path\nimport hashlib, json, os, shutil, subprocess, sys, tarfile\n\nINPUT = Path('/kaggle/input')\nWORK = Path('/kaggle/working')\nbundles = sorted(INPUT.glob('**/merenvay-arc2-offline-bundle.tar.gz'))\nif len(bundles) != 1:\n    raise RuntimeError(f'Expected exactly one MERENVAY bundle, found {len(bundles)}: {bundles}')\nbundle = bundles[0]\nchecks = sorted(INPUT.glob('**/merenvay-arc2-offline-bundle.tar.gz.sha256'))\nif len(checks) != 1:\n    raise RuntimeError(f'Expected exactly one bundle checksum file, found {len(checks)}: {checks}')\nexpected = checks[0].read_text(encoding='utf-8').strip().split()[0].lower()\nactual = hashlib.sha256(bundle.read_bytes()).hexdigest()\nif actual != expected:\n    raise RuntimeError(f'Bundle SHA-256 mismatch: expected {expected}, got {actual}')\nprint('Bundle SHA-256 verified:', actual)\n\nextract_root = WORK / 'merenvay_bundle_unpack'\nif extract_root.exists():\n    shutil.rmtree(extract_root)\nextract_root.mkdir(parents=True)\nroot_resolved = extract_root.resolve()\nwith tarfile.open(bundle, 'r:gz') as tf:\n    for member in tf.getmembers():\n        target = (extract_root / member.name).resolve()\n        if target != root_resolved and root_resolved not in target.parents:\n            raise RuntimeError(f'Unsafe archive path: {member.name}')\n    tf.extractall(extract_root)\n\npackage = extract_root / 'package'\nentry = package / 'app' / 'kaggle_entry.py'\nif not entry.is_file():\n    raise FileNotFoundError(entry)\nenv = os.environ.copy()\nenv['PYTHONHASHSEED'] = '0'\nenv['MERENVAY_PACKAGE_ROOT'] = str(package)\nenv['KAGGLE_INPUT_ROOT'] = str(INPUT)\nenv['KAGGLE_WORKING_ROOT'] = str(WORK)\nsubprocess.run([sys.executable, str(entry)], check=True, env=env)\n\nout = WORK / 'submission.json'\nif not out.exists() or out.stat().st_size == 0:\n    raise RuntimeError('submission.json was not produced')\ndata = json.loads(out.read_text(encoding='utf-8'))\nassert data and all(isinstance(v, list) and len(v) > 0 for v in data.values())\nassert all(all(isinstance(a, dict) and 'attempt_1' in a and 'attempt_2' in a for a in v) for v in data.values())\ndigest = hashlib.sha256(out.read_bytes()).hexdigest()\nprint('tasks:', len(data))\nprint('test_inputs:', sum(len(v) for v in data.values()))\nprint('submission_sha256:', digest)\nprint('output:', out)\nprint('CLAIM_BOUNDARY: official score only after Kaggle/ARC Prize evaluation.')\n